In [3]:
import os
import re
import time
import pandas as pd
import requests
from dotenv import load_dotenv

# === Load environment and GitHub tokens ===
env_path = "All_Tokens.env"
load_dotenv(env_path)
tokens = [os.getenv(f"GITHUB_TOKEN_{i}") for i in range(1, 7) if os.getenv(f"GITHUB_TOKEN_{i}")]
if not tokens:
    raise ValueError("❌ No GitHub tokens found.")
token_index = 0

# === CI Patterns ===
ci_patterns = {
    r'\.travis\.yml$': 'Travis CI',
    r'\.appveyor\.yml$': 'AppVeyor',
    r'appveyor\.yml$': 'AppVeyor',
    r'circle\.yml$': 'CircleCI',
    r'\.circleci/config\.yml$': 'CircleCI',
    r'azure-pipelines\.yml$': 'Azure Pipelines',
    r'\.github/workflows/.*\.(yml|yaml)$': 'GitHub Actions',
    r'bitbucket-pipelines\.yml$': 'Bitbucket',
    r'\.gitlab-ci\.yml$': 'GitLab',
    r'Jenkinsfile\.yml$': 'Jenkins',
    r'bitrise\.yml$': 'Bitrise',
    r'bamboo\.yml$': 'Bamboo',
    r'codeship-services\.yml$': 'Codeship',
    r'\.gocd\.yaml$': 'GoCD',
    r'\.cirrus\.yml$': 'Cirrus',
    r'wercker\.yaml$': 'Wercker',
    r'semaphore\.yml$': 'Semaphore',
    r'codemagic\.yaml$': 'Nevercode',
}
ci_types = sorted(set(ci_patterns.values()))

# === Paths ===
input_csv = r"C:\Android Mobile App\Step1_URL_Search\Type_1_Searching_Pipeline_July 14\step3_removal_keyword_output.csv"
output_csv = r"C:\Android Mobile App\Step1_URL_Search\Type_1_Searching_Pipeline_July 14\step4_ci_detection_output.csv"
yml_log_csv = r"C:\Android Mobile App\Step1_URL_Search\Type_1_Searching_Pipeline_July 14\step4_detected_yml_files.csv"

# === Load input data ===
input_df = pd.read_csv(input_csv)
input_df['html_url'] = input_df['html_url'].astype(str).str.strip()

# Filter: keep all rows but only process Valid_Repo_Step3 == yes
if os.path.exists(output_csv):
    print("🔁 Resuming from previously saved output.")
    input_df = pd.read_csv(output_csv)
else:
    input_df['yml_detected'] = input_df.get('yml_detected', 'none')
    input_df['total_yml_files'] = input_df.get('total_yml_files', 0)
    for ci in ci_types:
        if f"{ci}_count" not in input_df.columns:
            input_df[f"{ci}_count"] = 0

# === Load or initialize log file for matched ymls ===
if os.path.exists(yml_log_csv):
    yml_log = pd.read_csv(yml_log_csv)
    detected_files = yml_log.to_dict("records")
else:
    detected_files = []

# === Filter to valid repos and unprocessed ===
to_process = input_df[(input_df['Valid_Repo_Step3'].str.lower() == 'yes') & (input_df['yml_detected'] == 'none')].copy()
print(f"🔍 Starting detection: {len(to_process)} repos to review.")

def get_valid_response(url, headers):
    global token_index
    for _ in range(len(tokens)):
        token_id = token_index % len(tokens)
        headers['Authorization'] = f'token {tokens[token_id]}'
        response = requests.get(url, headers=headers)
        if response.status_code == 403 and response.headers.get("X-RateLimit-Remaining") == "0":
            reset_time = int(response.headers.get("X-RateLimit-Reset", time.time() + 60))
            wait_seconds = max(reset_time - int(time.time()), 1)
            print(f"⏳ Token {token_id+1} rate-limited. Sleeping for {wait_seconds} seconds...")
            time.sleep(wait_seconds)
            token_index += 1
            continue
        elif response.status_code in {200, 404}:
            return response, token_id + 1
        else:
            print(f"⚠️ Unexpected status code: {response.status_code} on URL: {url}")
            token_index += 1
            continue
    return None, None

# === Process repos ===
for review_counter, idx in enumerate(to_process.index, start=1):
    url = input_df.at[idx, 'html_url']
    print(f"🔎 [{review_counter}/{len(to_process)}] Checking: {url}")
    try:
        parts = url.rstrip('/').split('/')
        owner, repo = parts[-2], parts[-1]
        headers = {}

        r1, token_used = get_valid_response(f"https://api.github.com/repos/{owner}/{repo}", headers)
        if not r1 or r1.status_code != 200:
            print("❌ Repo info failed.")
            input_df.at[idx, 'yml_detected'] = 'no'
            continue

        default_branch = r1.json().get('default_branch', 'main')

        r2, _ = get_valid_response(f"https://api.github.com/repos/{owner}/{repo}/git/trees/{default_branch}?recursive=1", headers)
        if not r2 or r2.status_code != 200:
            print("❌ File tree failed.")
            input_df.at[idx, 'yml_detected'] = 'no'
            continue

        files = [item['path'] for item in r2.json().get('tree', []) if item['type'] == 'blob']
        matched = []
        for f in files:
            for pattern, ci_type in ci_patterns.items():
                if re.search(pattern, f, re.IGNORECASE):
                    raw_url = f"https://raw.githubusercontent.com/{owner}/{repo}/{default_branch}/{f}"
                    r3, _ = get_valid_response(raw_url, headers)
                    if r3 and r3.status_code == 200:
                        matched.append((f, ci_type))
                        detected_files.append({
                            "html_url": url,
                            "file_path": f,
                            "ci_type": ci_type,
                            "token_used": token_used
                        })
                    break

        input_df.at[idx, 'yml_detected'] = 'yes' if matched else 'no'
        input_df.at[idx, 'total_yml_files'] = len(matched)
        ci_counts = {}
        for _, ci in matched:
            ci_counts[ci] = ci_counts.get(ci, 0) + 1
        for ci in ci_types:
            input_df.at[idx, f"{ci}_count"] = ci_counts.get(ci, 0)

        input_df.to_csv(output_csv, index=False)
        pd.DataFrame(detected_files).to_csv(yml_log_csv, index=False)
    except Exception as e:
        print(f"⚠️ Error processing {url}: {e}")
        continue

print("\n✅ CI YML scan complete. Output and logs saved.")


🔍 Starting detection: 14690 repos to review.
🔎 [1/14690] Checking: https://github.com/Dawnthorn/nagare
🔎 [2/14690] Checking: https://github.com/bpellin/keepassdroid
🔎 [3/14690] Checking: https://github.com/connectbot/connectbot
🔎 [4/14690] Checking: https://github.com/JakeWharton/SMSMorse
🔎 [5/14690] Checking: https://github.com/JakeWharton/SMSBarrage
🔎 [6/14690] Checking: https://github.com/millenomi/diceshaker
🔎 [7/14690] Checking: https://github.com/pocmo/Yaaic
🔎 [8/14690] Checking: https://github.com/konklone/congress-android
🔎 [9/14690] Checking: https://github.com/johannilsson/sthlmtraveling
🔎 [10/14690] Checking: https://github.com/yaxim-org/yaxim
🔎 [11/14690] Checking: https://github.com/talklittle/reddit-is-fun
🔎 [12/14690] Checking: https://github.com/ProjectCCNx/ccnx
🔎 [13/14690] Checking: https://github.com/konklone/campyre
🔎 [14/14690] Checking: https://github.com/davidw/hecl
🔎 [15/14690] Checking: https://github.com/XCSoar/XCSoar
🔎 [16/14690] Checking: https://github.com/